In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.DataFrame([[8,8,4],[7,9,5],[6,10,6],[5,12,7]],columns=["cgpa","profile_score","lpa"])

In [4]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [11]:
def initilize_parameter(layer_dims):
    np.random.seed(3)
    parameters={}
    L = len(layer_dims)
    
    for l in range(1,L):
        parameters["W" + str(l)] = np.ones((layer_dims[l-1], layer_dims[l]))*0.1
        parameters["b" + str(l)] = np.zeros((layer_dims[l], 1))
        
    return parameters
        

In [12]:
initilize_parameter([2,2,1])

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

In [14]:
def linear_forward(A_prev,W,b):
    Z = np.dot(W.T,A_prev) + b
    return Z

In [81]:
def L_layer_forward(X,parameters):
    
    A = X
    L = len(parameters)//2
    
    for l in range(1,L+1):
        A_prev = A
        Wl = parameters["W" + str(l)]
        bl = parameters["b" + str(l)]
        
        # print("A" + str(l-1) + ": ", A_prev )
        # print("W" + str(l) + ": ", Wl )
        # print("b" + str(l) + ": ", bl)
        # print("--"*20)
        A = linear_forward(A_prev,Wl,bl)
        # print("A" + str(l) + ": ", A)
        # print("**"*20)
         
    return A, A_prev

In [51]:
parameters = initilize_parameter([2,2,1])

In [70]:
X = df[["cgpa","profile_score"]].values[2].reshape(2,1)
y = df[["lpa"]].values[2][0]

In [71]:
y_hat, A1 = L_layer_forward(X,parameters)

A0:  [[ 6]
 [10]]
W1:  [[0.22511092 0.24403404]
 [0.22593261 0.2450905 ]]
b1:  [[0.17506907]
 [0.        ]]
----------------------------------------
A1:  [[3.78506072]
 [3.91510926]]
****************************************
A1:  [[3.78506072]
 [3.91510926]]
W2:  [[0.13659405]
 [0.14255515]]
b2:  [[0.22720266]]
----------------------------------------
A2:  [[1.30233844]]
****************************************


In [72]:
y_hat = y_hat[0][0]

In [73]:
A1

array([[3.78506072],
       [3.91510926]])

In [74]:
def loss_calculation(y,y_hat):
    return (y-y_hat)**2

In [75]:
loss_calculation(y,y_hat)

np.float64(22.068024164813938)

In [76]:
def update_parameters(parameters,y,y_hat,A1,X, n):
    parameters["W1"][0][0] =  parameters["W1"][0][0] + (n * 2 * (y-y_hat) * parameters["W2"][0][0] * X[0][0])
    parameters["W1"][0][1] =  parameters["W1"][0][1] + (n * 2 * (y-y_hat) * parameters["W2"][0][0] * X[1][0])    
    parameters["b1"][0][0] =  parameters["b1"][0][0] + (n * 2 * (y-y_hat))    
    parameters["W1"][1][0] =  parameters["W1"][1][0] + (n * 2 * (y-y_hat) * parameters["W2"][1][0] * X[0][0])
    parameters["W1"][1][1] =  parameters["W1"][1][1] + (n * 2 * (y-y_hat) * parameters["W2"][1][0] * X[1][0])    
    parameters["b1"][0][0] =  parameters["b1"][0][0] + (n * 2 * (y-y_hat) * parameters["W2"][0][0])
    
    parameters["W2"][0][0] =  parameters["W2"][0][0] + (n * 2 * (y-y_hat) * parameters["W2"][0][0] * A1[0][0])
    parameters["W2"][1][0] =  parameters["W2"][1][0] + (n * 2 * (y-y_hat) * parameters["W2"][0][0] * A1[1][0])    
    parameters["b2"][0][0] =  parameters["W2"][1][0] + (n * 2 * (y-y_hat))    

In [77]:
update_parameters(parameters,y,y_hat,A1,X,0.01)

In [78]:
parameters

{'W1': array([[0.30211164, 0.37236857],
        [0.30629371, 0.37902567]]),
 'b1': array([[0.28185576],
        [0.        ]]),
 'W2': array([[0.18516945],
        [0.21066736]]),
 'b2': array([[0.30462059]])}

In [83]:
# epoch implementation 


parameters = initilize_parameter([2,2,1])
epochs = 5


for i in range(epochs):
    Loss = []
    
    for j in range(df.shape[0]):
        X = df[["cgpa","profile_score"]].values[j].reshape(2,1)
        y = df[["lpa"]].values[j][0]
        y_hat, A1 = L_layer_forward(X,parameters)
        y_hat = y_hat[0][0]
        Loss.append(loss_calculation(y,y_hat))
        update_parameters(parameters,y,y_hat,A1,X,0.01)
    
    print("Epoch - ", i, "Loss - ", np.array(Loss).mean())  

Epoch -  0 Loss -  18.073441113563085
Epoch -  1 Loss -  1.3732720618828413
Epoch -  2 Loss -  2.9659946512846704
Epoch -  3 Loss -  2.710225794917338
Epoch -  4 Loss -  2.4698195543300345


In [84]:
parameters

{'W1': array([[0.26582211, 0.62038932],
        [0.25889916, 0.79857807]]),
 'b1': array([[0.42111308],
        [0.        ]]),
 'W2': array([[0.17805794],
        [0.46589499]]),
 'b2': array([[0.47953544]])}

In [94]:
import tensorflow
from tensorflow.keras import Sequential
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [86]:
model = Sequential()

model.add(Dense(2,activation="linear",input_dim=2))
model.add(Dense(1,activation="linear"))

d:\Education\Extra\Artificial Intelligence\Deep-Learning\.venv_tf312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [87]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [93]:
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [90]:
new_weights = [np.array([[0.1 , 0.1],[0.1 , 0.1]], dtype= np.float32),
    np.array([0 , 0], dtype= np.float32),
    np.array([[0.1],[0.1]], dtype= np.float32),
    np.array([0 ], dtype= np.float32)]

In [91]:
new_weights

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [92]:
model.set_weights(new_weights)

In [96]:
optimizer = keras.optimizers.Adam(learning_rate= 0.001)
model.compile(loss="mean_squared_error",optimizer=optimizer)
model.fit(df.iloc[:,0:-1].values , df["lpa"].values, epochs = 75, verbose=1, batch_size= 1)

Epoch 1/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 27.9067  
Epoch 2/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 27.5874 
Epoch 3/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 27.2635 
Epoch 4/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 26.9028 
Epoch 5/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 26.5680 
Epoch 6/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 26.1994 
Epoch 7/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 25.8560 
Epoch 8/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 25.4791 
Epoch 9/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 25.0521 
Epoch 10/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 24.6772 
Epoch 11/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 24.2426 
Epoch 12/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 23.8156 
Epoch 13/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 23.4046 
Epoch 14/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 22.9327 
Epoch 15/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 22.4852 
Epoch 16/75
4/4 ━━

In [97]:
model.get_weights()

[array([[0.37365356, 0.37365356],
        [0.36547363, 0.36547363]], dtype=float32),
 array([0.27226415, 0.27226415], dtype=float32),
 array([[0.3728561],
        [0.3728561]], dtype=float32),
 array([0.20467512], dtype=float32)]